In [42]:
import pandas as pd
import numpy as np
import re

from extraction.texts import model_loader, text_encoding

In [2]:
df = pd.read_csv('data/texts/suicide_harm/suicide.csv')
df = df[["text", 'class']]

df['class'] = df['class'].replace({
                            "suicide":1,
                            "non-suicide":0
                        })

df = pd.concat([
    df[df["class"] == 0].head(10000),
    df[df["class"] == 1].head(40000)
]).reset_index(drop=True)

df["text"] = (
    df["text"].apply(lambda x: re.sub(r"[^A-Za-z0-9]", " ", 
                                             x, count=0, flags=0))
            .apply(lambda x: re.findall(r"[A-Za-z0-9]+", x))
            .apply(lambda x: " ".join(x))
)

/var/folders/tg/66tmmx452mg3qln33ts5xqh80000gn/T/ipykernel_88927/4066350544.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['class'] = df['class'].replace({


In [3]:
texts = df["text"].to_list()

In [4]:
model, model_code = model_loader('mps', model_code='e5')
text_vectors = text_encoding(texts, model, model_code)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [29]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [10]:
x, y = text_vectors, df["class"].to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    stratify=y,
                                                    test_size=0.2)

In [28]:
log_reg = LogisticRegression(
    penalty='l1',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='saga',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9732 (When flagged positive, accuracy is 97.32%)
Custom Recall Score:    0.9633 (Captured 96.33% of all true positive cases)
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9728 (When flagged positive, accuracy is 97.28%)
Custom Recall Score:    0.9608 (Captured 96.08% of all true positive cases)
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9744 (When flagged positive, accuracy is 97.44%)
Custom Recall Score:    0.9631 (Captured 96.31% of all true positive cases)
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9769 (When flagged positive, accuracy is 97.69%)
Custom Recall Score:    0.9633 (Captured 96.33% of all true positive cases)
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9770 (When flagged positive, accuracy is 97.70%)
Custom Recall Score:    0.96

In [33]:
log_reg = LogisticRegression(
    penalty='l1',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='saga',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

           0       0.86      0.91      0.89      2000
           1       0.98      0.96      0.97      8000

    accuracy                           0.95     10000
   macro avg       0.92      0.94      0.93     10000
weighted avg       0.95      0.95      0.95     10000

[[1820  180]
 [ 287 7713]]


In [34]:
import joblib

In [38]:
joblib.dump(log_reg, open("model/suicide.jobllib", 'wb'))